# Multiquery
using langchain retrieval, also local gemma 3:1b

In [138]:
#Imports and configuration
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
import langchain
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers import MultiQueryRetriever
import logging


In [139]:
# environmental variables
PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = DATA_DIR / "chromadb"

# winning chunking strategy collection
COLLECTION_NAME = "recursive_100"  # (misnamed, actually recursive_1000)
# LangChain wrapper around my local Ollama model for query reformulation. Qwen because it is good at following directions
llm = ChatOllama(model="gemma3:1b")
n_queries = 5

In [140]:
# Same embedding model as used during indexing, must match exactly
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"}
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6100.52it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [141]:
# LangChain wrapper around the ChromaDB collection

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    collection_name="recursive_100",  # misnamed, actually recursive_1000
    embedding_function=embedding_model
)
print(f"Vectorstore loaded: {vectorstore._collection.count()} chunks")

Vectorstore loaded: 31741 chunks


In [142]:
# LangChain wrapper around my local Ollama model for query reformulation. Qwen because it is good at following directions
llm = ChatOllama(model="gemma3:1b")
prompt_text = f"""You are an AI assistant helping to improve information retrieval
for a scientific database about algae research.

Your task is to generate {n_queries} alternative versions of the
given user question. Each alternative should approach the same
information need from a different angle or use different
terminology, so that together they can retrieve a broader set
of relevant documents. If the original question is not in English, first translate it to English before generating alternatives. All alternative questions must be in English.

Provide these alternative questions separated by newlines.
Do not number them. Do not add any explanation or preamble.
Only output the alternative questions.

Original question: {{question}}"""

multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template=prompt_text
) #originally 5 wasnt hardcoded, but it brought me problemas

In [143]:
# Wraps prompt in LangChain's PromptTemplate
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template=prompt_text
)

# Create the retriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),#5 is a good baseline
    llm=llm,
    prompt=multi_query_prompt,
    include_original=True  # also query with the original question
)

In [144]:
# Test ofthe retriever
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.DEBUG)

test_query = "Co je  Zostera Marina a kde roste?"

results = retriever.invoke(test_query)

print(f"\nRetrieved {len(results)} unique chunks")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200])


Retrieved 10 unique chunks

--- Chunk 1 ---
연구는 서해안 대표종인 지충이의 계절과 수심별 생물량 분

조하대 연구에서는 수심 1-2 m에서 다시마, 구멍갈파래, 지

포를 정량적으로 밝힌 최초의 연구이다. 해안 저층 구성과

누아리사촌(Grateloupia acuminata)이 서식하고 수심 4 m에서

큰 조석차로 인해 해수의 투명도가 낮은 서해안에서 지충이

는 거머리말(Zostera marin

--- Chunk 2 ---
Methanol crude extract of the sea grass Zostera marina L. and organic solvent fractions (n-hexane, chloroform, ethyl acetate, n-butanol, and water) were screened for antioxidant activity (total phenol

--- Chunk 3 ---
0.22 0.35 1.00 4 0.08 0.10 045 1.00 4 0.13 0.19 0.84 1.00 5 0.08 0.08 059 0.52 1.00 5 0.08 0.14 040 0.68 1.00 St. 1 2 3 4 5 St. 1 2 3 4 5 October November Je] SAHA. Ue VA 37} 404 OME SLA ve o] BA PSTS

--- Chunk 4 ---
et al. submitted). 현재 우리나라의 잘피 서식 면적은 대략적 으로 55-70 km2로 추정되고 있으며, 거머리말(Zostera marina)이 대부분(50-60 km2)을 차지하고 있다(Lee and Lee 2003). 우리나라 거의 모든 연안에 많은 양의 잘피가 분포하

고 있으며, 따라서 우리 연안의 탄소순환에 잘피가 매우 중

요한 역할

--- Chunk 5 ---
Statistical analyses were carried out using STATISTI- CA version 5.0 software. A on

In [145]:
response = llm.invoke(multi_query_prompt.format(question=test_query))
print(response.content)

Co je Zostera Marina a kde roste?
Co je Zostera Marina a kde roste?
What is Zostera Marina and where does it grow?
What is Zostera Marina and where does it thrive?
What is Zostera Marina and where can it be found?
What is Zostera Marina and what regions does it occur in?


# Reranking

In [146]:
from sentence_transformers import CrossEncoder

# Load the cross-encoder reranker
reranker = CrossEncoder("BAAI/bge-reranker-base")



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4897.95it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     | Details
--------------------------------+------------+--------
roberta.embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## TEST

In [147]:
# Score each retrieved chunk against the original query
pairs = [[test_query, doc.page_content] for doc in results]
scores = reranker.predict(pairs)

# Attaches scores and sorts
ranked = sorted(zip(scores, results), key=lambda x: x[0], reverse=True)

# Takes top 5. might adjust later
top_chunks = ranked[:5]

print(f"Reranked {len(results)} chunks, keeping top {top_k}:\n")
for score, doc in top_chunks:
    print(f"Score: {score:.4f}")
    print(doc.page_content[:550])
    print()

Reranked 10 chunks, keeping top 5:

Score: 0.2965
Zostera marina Linnaeus. Zostera marina is common in the Keret Archipelago. It occurs primarily is sites influ- enced by freshwater where it forms a band in the shallow subtidal immediately below Ascophyllum nodosum. In some shallow bays it forms more extensive populations, and it sometimes occurs in the intertidal zone when it is exposed during spring tides. While we collected leaves of Z. marina to examine for epiphytes, they were generally

http://dx.doi.org/10.4490/algae.2013.28.3.267

276

free of epiphytic macroalgae except for the ubiqui

Score: 0.0297
Shoots of Zostera marina were collected in the coastal waters of Anpori, Jangsuri and Wonpori in Gamak Bay, Yeosu, Jeollanamdo, Korea (Fig. 1) from December 1999 to November 2000.

Five morphological characteristics of the eelgrass such as shoot height, total shoot weight, leaf width, leaf number, branch number were measured every month.

Water temperature, pH and salinity were mea

# INTEGRATING GENERATION

In [148]:
#context
# Builds context with metadata for citation
context_parts = []
for i, (score, doc) in enumerate(top_chunks, 1):
    meta = doc.metadata
    header = f"[{i}] {meta.get('authors', 'Unknown')} ({meta.get('year', 'n.d.')}). {meta.get('title', 'Untitled')}"
    context_parts.append(f"{header}\n{doc.page_content}")

context = "\n\n".join(context_parts)

In [149]:
print(context[:5000])

[1] David J. Garbary, Elena R. Tarakhovskaya (2013). Marine macroalgae and associated flowering plants from the Keret Archipelago, White Sea
Zostera marina Linnaeus. Zostera marina is common in the Keret Archipelago. It occurs primarily is sites influ- enced by freshwater where it forms a band in the shallow subtidal immediately below Ascophyllum nodosum. In some shallow bays it forms more extensive populations, and it sometimes occurs in the intertidal zone when it is exposed during spring tides. While we collected leaves of Z. marina to examine for epiphytes, they were generally

http://dx.doi.org/10.4490/algae.2013.28.3.267

276

free of epiphytic macroalgae except for the ubiquitous skeins of Cladophora glomerata. Kalugina (1958) and Vozzhinskaya (1986) discuss community ecology of Z. marina in the White Sea.

[2] Do-Hoon Kim‘*, Jin-Hyung Park! and Jong-Ahm Shin (2004). A Preliminary Study on Growth and Habitat Characteristics of Zostera marina (Zostaceae) in Gamak Bay, Yeosu
Shoot

In [150]:
#Generation with reranked context

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Build context from reranked top chunks
context = "\n\n".join([doc.page_content for score, doc in top_chunks])

prompt = f"""
Role: You are an experienced marine biologist and algae cultivation specialist with deep expertise in seagrass ecology, microalgae biotechnology, 
and industrial algae applications.
You prioritize scientific precision, cite 
specific data when available, and clearly distinguish between established 
findings and uncertainties.
Task: Given retrieved passages from scientific literature on algae and 
marine biology, answer the user's question following these steps:

1. First, assess which of the provided passages are relevant to the question 
   and briefly note why.
2. Synthesize information from ALL relevant passages into a coherent answer.
   You must draw from multiple sources where possible; do not rely on a single passage.
3. If passages contain conflicting information, acknowledge the conflict and 
   explain both positions.
4. If the provided context is insufficient to fully answer the question, 
   state what is missing.
Domain constraint: Focus on the user's specific industry context.
Output format: Brief paragraph. You MUST cite every passage you use with its bracketed number [1], [2], etc. from the context headers. Use multiple citations to support your answer. Then at the end of your answer, list all sources of retrieved chunks by bracketed number, title, author, year

Keep your answer grounded strictly in the provided context. Do not introduce 
external knowledge beyond what is given. Match your answer's depth to the question's complexity: give concise 
answers to simple questions, detailed analysis to complex ones. Respond in the same language as the user's question.

Context: {context}

Question: {test_query}
Answer:"""

In [151]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Zostera marina je seagrass, tedy mořská tráva (eelgrass) [4]. V Keret Archipelagu roste převážně na místech ovlivněných sladkovodním průtokem a vytváří pás v mělké subtidální zóně těsně pod Ascophyllum nodosum; v některých mělčích zálivech může mít rozsáhlejší populace a může se objevit i v intertidální zóně, když je vystavena během spring tides [1]. V Koreji se nachází v intertidální i subtidální zóně do hloubky do 5 m (obvykle 1–5 m, průměr kolem 2.5 m); v Geoje Island je spolu s dalšími druhy Z. japonica a Z. caespitosa sbírána a popisována v různých zónách (GDa, GGa: intertidální, GJa: subtidální) a byla pozorována i částečná expozice při odlivu [6]. Celkové rozšíření v Koreji je rovněž uváděno odhadem plošné plochy eelgrass stanovišť kolem 55–70 km2, z čehož 50–60 km2 připadá na Z. marina [5]. Z těchto údajů vyplývá, že Zostera marina je široce adaptabilní na mělčích pobřežních stanovištích—jak v intertidálních, tak subtidálních zónách do hloubky kolem 5 m, často v blízkosti menší

In [152]:
#trying to integrate json metadata
for score, doc in top_chunks:
    print(doc.metadata)
    print()

{'filename': 'algae-2013-28-3-267.pdf', 'year': '2013', 'authors': 'David J. Garbary, Elena R. Tarakhovskaya', 'title': 'Marine macroalgae and associated flowering plants from the Keret Archipelago, White Sea'}

{'filename': 'algae-2004-19-1-049.pdf', 'authors': 'Do-Hoon Kim‘*, Jin-Hyung Park! and Jong-Ahm Shin', 'year': '2004', 'title': 'A Preliminary Study on Growth and Habitat Characteristics of Zostera marina (Zostaceae) in Gamak Bay, Yeosu'}

{'filename': 'algae-2009-24-3-179.pdf', 'year': '2009', 'title': 'Deep Learning for Image Recognition', 'authors': '...'}

{'year': '2008', 'filename': 'algae-2008-23-3-241.pdf', 'title': 'Growth Dynamics and Carbon Incorporation of the Seagrass, Zostera marina L. in Jindong Bay and Gamak Bay on the Southern Coast of Korea', 'authors': '김태환, 박상률, 김영균, 김종협, 김승현, 정익교, 이근섭'}

{'year': '2008', 'filename': 'algae-2008-23-1-075.pdf', 'title': 'Algae Volume 23(1): 75-81, ', 'authors': 'Algae'}

